# Check Drift between Train and Prod dataset created in Model Experimentation Phase

### Install alibi_detect library

In [1]:
import numpy as np
np.__version__

'1.26.4'

In [2]:
!pip install alibi alibi_detect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.1/522.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.5/381.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 MB 8.1 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.1.0
    Uninstalling pillow-11.1.0:
      Successfully uninstalled pillow-11.1.0
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully u

In [1]:
import alibi
from alibi_detect.cd import ChiSquareDrift, TabularDrift
from alibi_detect.saving import save_detector, load_detector

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

### Importing Train and Prod Data that we created during Model Experimentation to check the drift

In [3]:
saheart_train_df = pd.read_parquet( "train.parquet" )

In [8]:
saheart_prod_df = pd.read_parquet( "prod.parquet" )

In [4]:
saheart_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   row.names  322 non-null    int64  
 1   sbp        322 non-null    int64  
 2   tobacco    322 non-null    float64
 3   ldl        322 non-null    float64
 4   adiposity  322 non-null    float64
 5   famhist    322 non-null    object 
 6   typea      322 non-null    int64  
 7   obesity    320 non-null    float64
 8   alcohol    322 non-null    float64
 9   age        322 non-null    int64  
 10  chd        322 non-null    int64  
dtypes: float64(5), int64(5), object(1)
memory usage: 27.8+ KB


In [5]:
x_features = list(saheart_train_df.columns)

In [6]:
x_features

['row.names',
 'sbp',
 'tobacco',
 'ldl',
 'adiposity',
 'famhist',
 'typea',
 'obesity',
 'alcohol',
 'age',
 'chd']

#### Specify the index of the columns which are categorical features

In [9]:
cat_vars = [5,10]

### Split the dataset into two sets

**Note**: In this example, we are taking train and prod split just as an example

In [10]:
X_train = saheart_train_df.copy()
X_prod = saheart_prod_df.copy()

In [11]:
categories_per_feature = {f: None for f in cat_vars}

In [12]:
categories_per_feature

{5: None, 10: None}

### Measure the drift

In [14]:
cd = TabularDrift(X_train.values,
                  p_val=.05,
                  categories_per_feature=categories_per_feature)

In [15]:
filepath = 'saheartdrift'  # change to directory where detector is saved
save_detector(cd, filepath, legacy = True)

In [16]:
cd = load_detector(filepath)

In [17]:
preds = cd.predict(X_prod.to_numpy())

### Printing the test results

- KS test for the numerical features
- chi-squared test for the categorical features

In [18]:
for f in range(cd.n_features):
    stat = 'Chi2' if f in list(categories_per_feature.keys()) else 'K-S'
    fname = x_features[f]
    stat_val, p_val = preds['data']['distance'][f], preds['data']['p_val'][f]
    print(f'{fname} -- {stat} {stat_val:.3f} -- p-value {p_val:.3f}')

row.names -- K-S 0.195 -- p-value 0.077
sbp -- K-S 0.130 -- p-value 0.458
tobacco -- K-S 0.198 -- p-value 0.069
ldl -- K-S 0.169 -- p-value 0.169
adiposity -- K-S 0.172 -- p-value 0.158
famhist -- Chi2 0.075 -- p-value 0.785
typea -- K-S 0.080 -- p-value 0.936
obesity -- K-S nan -- p-value nan
alcohol -- K-S 0.148 -- p-value 0.301
age -- K-S 0.196 -- p-value 0.075
chd -- Chi2 3.338 -- p-value 0.068


# *Here, No variable has p-value < 0.05. Hence, no drift has been observed*